In [ ]:
!pip install -q kailash kailash-ml polars torch torchvision matplotlib python-dotenv torch-geometric


# MLFP05 — Exercise 6.5: GNN Architecture Comparison


**WHAT YOU'LL LEARN:**
- Systematic comparison: GCN vs GAT vs GraphSAGE on the same dataset
- Metrics that matter: accuracy, convergence speed, parameter count
- When to choose each architecture (decision framework)
- Register the best model in kailash-ml ModelRegistry
- Visualise side-by-side training curves and embedding quality


**PREREQUISITES: M5/ex_6.1 (GCN), M5/ex_6.2 (GAT), M5/ex_6.3 (GraphSAGE).**

**ESTIMATED TIME: ~25 min**


In [ ]:
from __future__ import annotations

import asyncio
import pickle

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from shared.mlfp05.ex_6 import (
    OUTPUT_DIR,
    device,
    load_graph_data,
    plot_node_embeddings,
    plot_training_curves,
    register_model,
    setup_engines,
    train_node_classifier,
    viz,
)
from kailash_ml.types import MetricSpec

import matplotlib.pyplot as plt


## SETUP — Load Data and Build All Three Architectures


In [ ]:
print("=" * 70)
print("  GNN Architecture Comparison: GCN vs GAT vs GraphSAGE")
print("=" * 70)

graph_data = load_graph_data()
conn, tracker, exp_name, registry, has_registry = setup_engines()

X = graph_data["X"]
A = graph_data["A"]
A_norm = graph_data["A_norm"]
y = graph_data["y"]
y_np = graph_data["y_np"]
A_np = graph_data["A_np"]
N = graph_data["N"]
F_dim = graph_data["F_dim"]
n_classes = graph_data["n_classes"]
dataset_name = graph_data["dataset_name"]

HIDDEN_DIM = 16 if dataset_name == "Karate Club" else 64
EPOCHS = 100
SAMPLE_K = 10


### GCN ─────────────────────────────────────────────────────────────


In [ ]:
# TODO: Implement GCNLayer and GCN (same as exercise 6.1)
class GCNLayer(nn.Module):
    def __init__(self, in_dim: int, out_dim: int):
        super().__init__()
        # TODO: self.W = nn.Linear(in_dim, out_dim, bias=True)
        pass

    def forward(self, h: torch.Tensor, a_norm: torch.Tensor) -> torch.Tensor:
        # TODO: return a_norm @ self.W(h)
        pass


class GCN(nn.Module):
    def __init__(self, in_dim: int, hidden_dim: int, n_classes: int):
        super().__init__()
        # TODO: self.l1 = GCNLayer(in_dim, hidden_dim)
        #       self.l2 = GCNLayer(hidden_dim, n_classes)
        pass

    def forward(self, h: torch.Tensor, a_norm: torch.Tensor) -> torch.Tensor:
        # TODO: ReLU(l1) -> dropout(0.5) -> l2
        pass

    def embed(self, h: torch.Tensor, a_norm: torch.Tensor) -> torch.Tensor:
        # TODO: return F.relu(self.l1(h, a_norm))
        pass


### GAT ─────────────────────────────────────────────────────────────


In [ ]:
# TODO: Implement GATLayer and GAT (same as exercise 6.2)
class GATLayer(nn.Module):
    def __init__(self, in_dim: int, out_dim: int):
        super().__init__()
        # TODO: self.W, self.a_src, self.a_dst — all nn.Linear, no bias
        pass

    def forward(self, h: torch.Tensor, adj: torch.Tensor) -> torch.Tensor:
        # TODO: Project, compute attention scores, mask, softmax, aggregate
        # Wh = self.W(h)
        # e_src = self.a_src(Wh); e_dst = self.a_dst(Wh)
        # scores = F.leaky_relu(e_src + e_dst.T, negative_slope=0.2)
        # mask = adj + torch.eye(adj.size(0), device=adj.device)
        # scores = scores.masked_fill(mask == 0, float("-inf"))
        # alpha = F.softmax(scores, dim=1)
        # return alpha @ Wh
        pass


class GAT(nn.Module):
    def __init__(self, in_dim: int, hidden_dim: int, n_classes: int):
        super().__init__()
        # TODO: self.l1 = GATLayer(in_dim, hidden_dim)
        #       self.l2 = GATLayer(hidden_dim, n_classes)
        pass

    def forward(self, h: torch.Tensor, adj: torch.Tensor) -> torch.Tensor:
        # TODO: ELU(l1) -> dropout(0.5) -> l2
        pass

    def embed(self, h: torch.Tensor, adj: torch.Tensor) -> torch.Tensor:
        # TODO: return F.elu(self.l1(h, adj))
        pass


### GraphSAGE ───────────────────────────────────────────────────────


In [ ]:
# TODO: Implement GraphSAGELayer and GraphSAGE (same as exercise 6.3)
class GraphSAGELayer(nn.Module):
    def __init__(self, in_dim: int, out_dim: int, sample_k: int = 10):
        super().__init__()
        # TODO: self.W_self, self.W_neigh — nn.Linear, no bias
        #       self.sample_k = sample_k
        pass

    def forward(self, h: torch.Tensor, adj: torch.Tensor) -> torch.Tensor:
        n = h.size(0)
        # TODO: Neighbour sampling (training only), mean aggregation, combine
        # if self.training and self.sample_k < n:
        #     sample_mask = torch.zeros_like(adj)
        #     for i in range(n):
        #         neigh_idx = torch.where(adj[i] > 0)[0]
        #         if len(neigh_idx) <= self.sample_k:
        #             sample_mask[i, neigh_idx] = 1.0
        #         else:
        #             perm = torch.randperm(len(neigh_idx), device=h.device)[:self.sample_k]
        #             sample_mask[i, neigh_idx[perm]] = 1.0
        #         adj_sampled = sample_mask
        # else:
        #     adj_sampled = adj
        # deg_sampled = adj_sampled.sum(dim=1, keepdim=True).clamp(min=1.0)
        # h_neigh = (adj_sampled @ h) / deg_sampled
        # return self.W_self(h) + self.W_neigh(h_neigh)
        pass


class GraphSAGE(nn.Module):
    def __init__(
        self, in_dim: int, hidden_dim: int, n_classes: int, sample_k: int = 10
    ):
        super().__init__()
        # TODO: self.l1 = GraphSAGELayer(in_dim, hidden_dim, sample_k)
        #       self.l2 = GraphSAGELayer(hidden_dim, n_classes, sample_k)
        pass

    def forward(self, h: torch.Tensor, adj: torch.Tensor) -> torch.Tensor:
        # TODO: ReLU(l1) -> dropout(0.5) -> l2
        pass

    def embed(self, h: torch.Tensor, adj: torch.Tensor) -> torch.Tensor:
        # TODO: return F.relu(self.l1(h, adj))
        pass


## TRAIN — All Three Architectures Under Identical Conditions


In [ ]:
print(f"\n== Training all three architectures on {dataset_name} ==\n")

# GCN
print(f"--- GCN ---")
gcn = GCN(in_dim=F_dim, hidden_dim=HIDDEN_DIM, n_classes=n_classes)
gcn_losses, gcn_val, gcn_test = train_node_classifier(
    gcn,
    "GCN",
    A_norm,
    graph_data,
    tracker,
    exp_name,
    EPOCHS,
)

# GAT
print(f"\n--- GAT ---")
gat = GAT(in_dim=F_dim, hidden_dim=HIDDEN_DIM, n_classes=n_classes)
gat_losses, gat_val, gat_test = train_node_classifier(
    gat,
    "GAT",
    A,
    graph_data,
    tracker,
    exp_name,
    EPOCHS,
)

# GraphSAGE
print(f"\n--- GraphSAGE ---")
sage = GraphSAGE(
    in_dim=F_dim, hidden_dim=HIDDEN_DIM, n_classes=n_classes, sample_k=SAMPLE_K
)
sage_losses, sage_val, sage_test = train_node_classifier(
    sage,
    "GraphSAGE",
    A,
    graph_data,
    tracker,
    exp_name,
    EPOCHS,
)


## COMPARE — Quantitative Metrics Table


In [ ]:
print("\n" + "=" * 70)
print("  QUANTITATIVE COMPARISON")
print("=" * 70)

results = {
    "GCN": {
        "val_accs": gcn_val,
        "test_accs": gcn_test,
        "losses": gcn_losses,
        "model": gcn,
        "forward_arg": A_norm,
    },
    "GAT": {
        "val_accs": gat_val,
        "test_accs": gat_test,
        "losses": gat_losses,
        "model": gat,
        "forward_arg": A,
    },
    "GraphSAGE": {
        "val_accs": sage_val,
        "test_accs": sage_test,
        "losses": sage_losses,
        "model": sage,
        "forward_arg": A,
    },
}

# TODO: Print comparison table
# For each model, compute:
#   - n_params: sum(p.numel() for p in model.parameters())
#   - best_val: max(val_accs)
#   - best_test: max(test_accs)
#   - final_loss: losses[-1]
#   - conv_epoch: first epoch where val_acc >= 0.9 * best_val
# Hint: conv_epoch = next((i+1 for i, a in enumerate(val_accs) if a >= 0.9*best_val), EPOCHS)
print(
    f"\n{'Model':>12} {'Params':>8} {'Best Val':>10} {'Best Test':>10} "
    f"{'Final Loss':>12} {'Conv@90%':>10}"
)
print("-" * 66)

for name, r in results.items():
    n_params = sum(p.numel() for p in r["model"].parameters())
    best_val = max(r["val_accs"])
    best_test = max(r["test_accs"])
    final_loss = r["losses"][-1]
    # TODO: Compute convergence epoch (first epoch reaching 90% of best val accuracy)
    threshold = 0.9 * best_val
    conv_epoch = next(
        (i + 1 for i, a in enumerate(r["val_accs"]) if a >= threshold),
        EPOCHS,
    )
    print(
        f"{name:>12} {n_params:>8,} {best_val:>10.4f} {best_test:>10.4f} "
        f"{final_loss:>12.4f} {conv_epoch:>10}"
    )

# Determine the best model by best validation accuracy
best_name = max(results, key=lambda k: max(results[k]["val_accs"]))
best_model_obj = results[best_name]["model"]
best_val_acc = max(results[best_name]["val_accs"])
best_test_acc = max(results[best_name]["test_accs"])
print(f"\nBest model by validation accuracy: {best_name} ({best_val_acc:.4f})")


### Comparison Checkpoint ───────────────────────────────────────────


In [ ]:
assert len(results) == 3, "Should have results for all 3 architectures"
assert all(
    max(r["val_accs"]) > 0.3 for r in results.values()
), "All models should achieve > 30% accuracy (well above random for 7 classes)"
# INTERPRETATION: The comparison reveals architectural trade-offs:
# - GCN is simplest (fewest params) but uses fixed aggregation weights
# - GAT adds learnable attention but costs more parameters
# - GraphSAGE separates self vs neighbour projections and uses sampling
# Cora is a homogeneous citation graph where all three tend to perform
# similarly. Differences become more pronounced on heterogeneous graphs.
print("\n--- Comparison checkpoint passed --- all models evaluated\n")


## VISUALISE — Side-by-Side Training Curves and Embeddings


In [ ]:
print("=" * 70)
print("  VISUALISATIONS — Side-by-Side Comparison")
print("=" * 70)

# Plot 1: Loss curves overlay
plot_training_curves(
    metrics_dict={
        "GCN train loss": gcn_losses,
        "GAT train loss": gat_losses,
        "GraphSAGE train loss": sage_losses,
    },
    title="GNN Training Loss Comparison",
    y_label="Cross-Entropy Loss",
    filename="comparison_loss_curves.html",
)

# Plot 2: Accuracy curves overlay
plot_training_curves(
    metrics_dict={
        "GCN val acc": gcn_val,
        "GAT val acc": gat_val,
        "GraphSAGE val acc": sage_val,
    },
    title="GNN Validation Accuracy Comparison",
    y_label="Validation Accuracy",
    filename="comparison_accuracy_curves.html",
)

# TODO: Create side-by-side embedding visualisations (1 row, 3 columns)
# For each model:
# 1. Extract embeddings via model.embed()
# 2. PCA to 2D using SVD
# 3. Scatter plot coloured by class
# 4. Title with model name, val accuracy, and param count
# Hint: emb_centered = emb - emb.mean(axis=0, keepdims=True)
#        U, S, Vt = np.linalg.svd(emb_centered, full_matrices=False)
#        coords = emb_centered @ Vt.T[:, :2]
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for idx, (name, r) in enumerate(results.items()):
    model = r["model"]
    model.eval()
    with torch.no_grad():
        if name == "GCN":
            emb = model.embed(X, A_norm).cpu().numpy()
        else:
            emb = model.embed(X, A).cpu().numpy()

    # TODO: PCA to 2D and scatter plot for each model
    # emb_centered = emb - emb.mean(axis=0, keepdims=True)
    # U, S, Vt = np.linalg.svd(emb_centered, full_matrices=False)
    # coords = emb_centered @ Vt.T[:, :2]
    # cmap = plt.cm.get_cmap("tab10", n_classes)
    # for c in range(n_classes):
    #     mask = y_np == c
    #     axes[idx].scatter(coords[mask, 0], coords[mask, 1], c=[cmap(c)], s=8, alpha=0.5, label=f"Class {c}")
    # n_params = sum(p.numel() for p in model.parameters())
    # best_v = max(r["val_accs"])
    # axes[idx].set_title(f"{name}\nval={best_v:.3f}, params={n_params:,}")
    pass

fig.suptitle(
    f"Node Embedding Comparison — {dataset_name}",
    fontsize=15,
    fontweight="bold",
)
plt.tight_layout()
filepath = OUTPUT_DIR / "comparison_embeddings.png"
fig.savefig(filepath, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"  Saved: {filepath}")

# TODO: Create parameter efficiency plot (scatter: params vs accuracy)
# x-axis: number of parameters per model
# y-axis: best validation accuracy
# Annotate each point with the model name
fig, ax = plt.subplots(1, 1, figsize=(8, 6))
model_names = list(results.keys())
param_counts = [
    sum(p.numel() for p in results[n]["model"].parameters()) for n in model_names
]
best_vals = [max(results[n]["val_accs"]) for n in model_names]
colors = ["#2196F3", "#FF9800", "#4CAF50"]

# TODO: Scatter plot with annotations
# ax.scatter(param_counts, best_vals, s=200, c=colors, zorder=3, edgecolor="white", linewidth=2)
# for i, name in enumerate(model_names):
#     ax.annotate(name, (param_counts[i], best_vals[i]), textcoords="offset points", xytext=(10, 10))
# ax.set_xlabel("Number of Parameters")
# ax.set_ylabel("Best Validation Accuracy")
# ax.set_title("Parameter Efficiency: Accuracy vs Model Size")
# ax.grid(True, alpha=0.3)

plt.tight_layout()
filepath = OUTPUT_DIR / "comparison_param_efficiency.png"
fig.savefig(filepath, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"  Saved: {filepath}")

print("\n--- Visualisations saved ---\n")


## DECISION FRAMEWORK — When to Use Each Architecture


In [ ]:
print("=" * 70)
print("  DECISION FRAMEWORK: Choosing a GNN Architecture")
print("=" * 70)
print(
    """
  ARCHITECTURE SELECTION GUIDE:

  ┌─────────────────┬───────────────────────────────────────────────────┐
  │ Choose GCN when │ - Graph is small-to-medium (< 100K nodes)        │
  │                 │ - Neighbours are roughly equally important        │
  │                 │ - You want the simplest, fastest baseline         │
  │                 │ - Interpretability of attention is NOT needed     │
  │                 │ - Example: citation networks, molecular graphs    │
  ├─────────────────┼───────────────────────────────────────────────────┤
  │ Choose GAT when │ - Edge importance varies (some neighbours matter │
  │                 │   more than others)                               │
  │                 │ - You need INTERPRETABLE attention weights        │
  │                 │ - Regulated domain (finance, healthcare) needs    │
  │                 │   explainable model decisions                     │
  │                 │ - Example: fraud detection, drug interaction      │
  ├─────────────────┼───────────────────────────────────────────────────┤
  │ Choose          │ - Graph is large (100K+ nodes) — need sampling   │
  │ GraphSAGE when  │ - New nodes arrive at inference time (INDUCTIVE) │
  │                 │ - Mini-batch training required (GPU memory bound) │
  │                 │ - Example: social networks, recommendations,     │
  │                 │   dynamic knowledge graphs                       │
  └─────────────────┴───────────────────────────────────────────────────┘

  ACROSS ALL OF MODULE 5 — Architecture Selection Guide:

  ┌──────────────────┬──────────────────────────────────────────────────┐
  │ Data Type        │ Architecture                                     │
  ├──────────────────┼──────────────────────────────────────────────────┤
  │ Images           │ CNN / ViT + transfer learning (ImageNet)         │
  │ Text             │ Transformer + transfer learning (BERT / GPT)     │
  │ Sequences        │ LSTM / Transformer                               │
  │ Graphs           │ GNN (GCN / GAT / GraphSAGE — task dependent)    │
  │ Tabular          │ Gradient boosting (fast, reliable, no pretrain)  │
  └──────────────────┴──────────────────────────────────────────────────┘
"""
)


## REGISTER — Best Model in ModelRegistry


In [ ]:
print("=" * 70)
print("  MODEL REGISTRATION")
print("=" * 70)

if has_registry:
    version = register_model(
        registry=registry,
        name=f"m5_gnn_{best_name.lower()}",
        model=best_model_obj,
        metrics=[
            MetricSpec(name="best_val_accuracy", value=best_val_acc),
            MetricSpec(name="best_test_accuracy", value=best_test_acc),
            MetricSpec(name="final_loss", value=results[best_name]["losses"][-1]),
            MetricSpec(name="hidden_dim", value=float(HIDDEN_DIM)),
            MetricSpec(name="epochs", value=float(EPOCHS)),
        ],
    )
    print(f"  Registered {best_name}: version={version.version}")
    print(f"    val_acc={best_val_acc:.4f}, test_acc={best_test_acc:.4f}")
else:
    print("  ModelRegistry not available — skipping registration")


## FINAL SUMMARY


In [ ]:
print("\n" + "=" * 70)
print("  FINAL SUMMARY")
print("=" * 70)
print(
    f"\nDataset: {dataset_name} ({N} nodes, {graph_data['n_edges_undirected']} edges, "
    f"{n_classes} classes)"
)
print(f"\nNode Classification (best validation accuracy):")
for name, r in results.items():
    n_params = sum(p.numel() for p in r["model"].parameters())
    print(
        f"  {name:>12}: val={max(r['val_accs']):.4f}  "
        f"test={max(r['test_accs']):.4f}  params={n_params:,}"
    )
print(f"\nBest model: {best_name}")


## REFLECTION


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED — Exercise 6 Complete")
print("=" * 70)
print(
    f"""
  GNN ARCHITECTURES:
  [x] GCN: message passing as matrix multiplication  H' = A_norm @ H @ W
      Fixed aggregation via degree-normalised adjacency. Simplest and
      fastest. Works well on homogeneous graphs like citation networks.
  [x] GAT: learned attention weights over neighbours
      Each node decides how much to attend to each neighbour based on
      feature content. More expressive but more parameters. Interpretable.
  [x] GraphSAGE: sample + aggregate with separate self/neighbour projections
      INDUCTIVE — can generalise to unseen nodes at inference time.
      Neighbourhood sampling provides regularisation and scalability.

  GNN TASKS:
  [x] Node classification: predict the label of each node from its
      features and neighbourhood structure ({n_classes} classes on {dataset_name})
  [x] Link prediction: predict missing edges using dot-product decoder
      on GNN embeddings (foundation for recommendation systems)
  [x] Embedding visualisation: 2-D PCA projection shows class separation
      in the learned representation space

  ML ENGINEERING:
  [x] Tracked every GNN variant with ExperimentTracker (params, per-epoch
      loss, validation accuracy, test accuracy across {EPOCHS} epochs)
  [x] Registered best model in ModelRegistry with versioned metrics
  [x] Quantitative comparison table: parameters, accuracy, convergence
      speed — not eyeballing, but systematic tracked experiments

  SINGAPORE APPLICATIONS:
  [x] NUS/NTU research classification (GCN — citation graph)
  [x] PayNow/NETS fraud detection (GAT — interpretable attention)
  [x] GrabFood/foodpanda recommendations (GraphSAGE — scalable, inductive)
  [x] SGH drug-disease interaction discovery (link prediction)

  KEY INSIGHT: All three GNNs learn by aggregating information from
  neighbours, but they differ in HOW they aggregate:
    GCN        -> fixed weights (degree normalisation)
    GAT        -> learned weights (attention) + interpretability
    GraphSAGE  -> sampled + learned (separate self/neighbour) + inductive

  Next: In Exercise 7, you'll apply transfer learning with a pre-trained
  ResNet-18 to a new image classification task and export to ONNX...
"""
)

# Clean up
await conn.close()
